# A lightning fast intro to einsum

In [19]:
import torch
import numpy as np

## Basic operations with einsum

The core syntax with ``torch.einsum`` is "'indices on LHS -> indices on RHS', [list of tensors involved]"

In [2]:
# Initialise some tensors

A = torch.arange(6).reshape(2,3)  # (2,3) matrix
v = torch.arange(1,4)  # 3-dim row vector
w = torch.arange(3,6)  # 3-dim row vector
B = torch.tensor([[9,2], [6,4], [10,1]])  # (3,2) matrix

M = torch.tensor([[3,6,1],[7,2,4], [8,9,5]])  # (3,3) matrix

eps = torch.zeros((3, 3, 3)) # Levi-Civita symbol in 3-dim
eps[0, 1, 2] = eps[1, 2, 0] = eps[2, 0, 1] = 1
eps[0, 2, 1] = eps[2, 1, 0] = eps[1, 0, 2] = -1

In [3]:
# Transpose

torch.einsum('ij->ji', A) 

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [4]:
# Trace. Could also do torch.trace(M)

torch.einsum('ii->', M) 

tensor(10)

In [5]:
# Sum over rows (adding up values in each column to give one entry per column)

torch.einsum('ij->j', A)

tensor([3, 5, 7])

In [6]:
# Sum over columns (adding up values in each row to give one entry per row)

torch.einsum('ij->i', A)

tensor([ 3, 12])

In [7]:
# Hadamard product of two vectors

torch.einsum('i,i->i', [v,w])

tensor([ 3,  8, 15])

In [8]:
# Dot product 

torch.einsum('i,i->', v,w)

tensor(26)

In [9]:
# Outer product of two vectors

torch.einsum('i,j->ij', [w,v])

tensor([[ 3,  6,  9],
        [ 4,  8, 12],
        [ 5, 10, 15]])

In [10]:
# Matrix vector multiplication

torch.einsum('ij,j->i',[A,v])

tensor([ 8, 26])

In [11]:
# Matrix multiplication; torch.matmul(A,B)

torch.einsum('ij,jk -> ik', [A,B])

tensor([[ 26,   6],
        [101,  27]])

## More general operations

In [ ]:
# Batch Matrix Multiplication (BMM)
# Essentially, element-wise matrix multiplication of a row/list of matrices
# This is what you need to make your manual MLP training faster

a = torch.randn(3,2,5) # [(2x5), (2x5), (2x5)]
b = torch.randn(3,5,3) # [(5x3), (5x3), (5x3)]
torch.einsum('ijk,ikl->ijl', [a, b]) # Output is a row containing 3 (2x3) matrices as expected

tensor([[[ 3.0898,  0.1075,  0.2580],
         [ 4.3879,  1.2626,  1.0198]],

        [[-0.8362,  1.2100, -0.4684],
         [ 0.2213, -0.2911, -0.6054]],

        [[-0.4026,  0.6656, -0.1485],
         [-1.7784, -2.1633, -1.1964]]])

In [ ]:
# Contractions with more indices

a = torch.randn(2,3,5,7)
b = torch.randn(11,13,3,17,5)
result = torch.einsum('pqrs,tuqvr->pstuv', [a, b])
# print(result) # Very large, best not to print and understand what's happening with the resulting shape instead
print(result.shape)

torch.Size([2, 7, 11, 13, 17])


In [14]:
# Operations with multiple tensors (note: stuff like this or Hadamard products aren't part of traditional Einstein summation conventions
# but they fit the einsum framework)

a = torch.randn(2,3)
b = torch.randn(5,3,7)
c = torch.randn(2,7)
result = torch.einsum('ik,jkl,il->ij', [a, b, c])
print(result)
print(result.shape)

tensor([[-1.1929, -2.3193, -2.1041, 12.7429, -2.6570],
        [-1.5786,  1.0616, -1.7877,  1.0658, -2.4527]])
torch.Size([2, 5])


## A slightly involved example: determinant of square matrices

In [ ]:
# For some reason, this and other operations require that all tensors contain floats and not integers (int64)
torch.einsum('ijk,i,j,k->', [eps.float(), M[0].float(), M[1].float(), M[2].float()])

tensor(-49.)